# 05 · Uruguay frente a la región

**Fuente única:** API del Banco Mundial (World Development Indicators). Usar la misma fuente para todos los países garantiza que las cifras sean comparables: misma definición y misma metodología.

**Países:** toda Sudamérica (Uruguay, Argentina, Bolivia, Brasil, Chile, Colombia, Ecuador, Paraguay, Perú y Venezuela), Costa Rica y México, más dos referencias: el promedio de **América Latina y el Caribe** y la **OCDE**.

| Código | Indicador | Fuente original |
|---|---|---|
| NY.GDP.PCAP.PP.KD | PIB per cápita, PPA (US$ internacionales constantes de 2021) | Banco Mundial / ICP |
| NY.GDP.MKTP.KD.ZG | Crecimiento del PIB (% anual) | Cuentas nacionales |
| FP.CPI.TOTL.ZG | Inflación, precios al consumidor (% anual) | FMI |
| SL.UEM.TOTL.ZS | Desempleo total (% de la fuerza laboral) | OIT, estimación modelada |
| SL.UEM.1524.ZS | Desempleo juvenil, 15–24 años (%) | OIT, estimación modelada |
| SI.POV.GINI | Índice de Gini | Banco Mundial, PIP |
| SI.POV.UMIC | Pobreza con línea de US$ 8,30 por día (PPA 2021, % de la población) | Banco Mundial, PIP |
| VC.IHR.PSRC.P5 | Homicidios intencionales cada 100.000 habitantes | UNODC |
| SE.SEC.CUAT.UP.ZS | Población de 25+ años con secundaria completa (%) | UNESCO |

## 1. Descarga (bronze): un JSON por indicador en `raw/banco_mundial`

In [0]:
import os, json, urllib.request

RAW = "/Volumes/workspace/pobreza/raw/banco_mundial"
os.makedirs(RAW, exist_ok=True)

PAISES = ["URY", "ARG", "BOL", "BRA", "CHL", "COL", "ECU", "PRY", "PER", "VEN", "CRI", "MEX", "LCN", "OED"]
INDICADORES = ["NY.GDP.PCAP.PP.KD", "NY.GDP.MKTP.KD.ZG", "FP.CPI.TOTL.ZG", "SL.UEM.TOTL.ZS",
               "SL.UEM.1524.ZS", "SI.POV.GINI", "SI.POV.UMIC", "VC.IHR.PSRC.P5", "SE.SEC.CUAT.UP.ZS"]

for ind in INDICADORES:
    url = (f"https://api.worldbank.org/v2/country/{';'.join(PAISES)}/indicator/{ind}"
           f"?format=json&date=2000:2025&per_page=5000")
    with urllib.request.urlopen(url) as r:
        datos = json.loads(r.read())
    with open(f"{RAW}/{ind}.json", "w") as f:
        json.dump(datos[1], f)
    print(ind, len(datos[1]), "filas")

In [0]:
from pyspark.sql import functions as F

filas = []
for ind in INDICADORES:
    for r in json.load(open(f"{RAW}/{ind}.json")):
        filas.append((ind, r["indicator"]["value"], r["countryiso3code"], r["country"]["value"], r["date"], None if r["value"] is None else float(r["value"])))

bronze = spark.createDataFrame(filas, "indicador string, indicador_nombre string, iso3 string, pais_en string, anio string, valor double")
bronze.write.mode("overwrite").option("overwriteSchema", True).saveAsTable("workspace.pobreza.banco_mundial_bronze")
print(bronze.count(), "filas")

## 2. Silver

- Se sacan los años sin dato.
- Nombres de países en español.
- El año pasa a entero y el valor se redondea a 2 decimales.

In [0]:
NOMBRES = {"URY": "Uruguay", "ARG": "Argentina", "BRA": "Brasil", "CHL": "Chile", "PRY": "Paraguay",
           "CRI": "Costa Rica", "MEX": "México", "BOL": "Bolivia", "COL": "Colombia",
           "ECU": "Ecuador", "PER": "Perú", "VEN": "Venezuela", "LCN": "América Latina y el Caribe", "OED": "OCDE"}
mapa = F.create_map([F.lit(x) for kv in NOMBRES.items() for x in kv])

silver = (spark.table("workspace.pobreza.banco_mundial_bronze")
    .filter(F.col("valor").isNotNull())
    .select("indicador", "iso3", mapa[F.col("iso3")].alias("pais"),
            F.col("anio").cast("int").alias("anio"), F.round("valor", 2).alias("valor")))
silver.write.mode("overwrite").option("overwriteSchema", True).saveAsTable("workspace.pobreza.banco_mundial_silver")
display(silver.groupBy("indicador").agg(F.count("*").alias("filas"), F.max("anio").alias("ultimo_anio")))

## 3. Gold + export

In [0]:
import os

def carpeta_data():
    ruta = os.path.dirname(dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get())
    if not ruta.startswith("/Workspace"):
        ruta = "/Workspace" + ruta
    if os.path.basename(ruta) == "notebooks":   # si estoy en la carpeta notebooks...
        ruta = os.path.dirname(ruta)            # ...subo un nivel, a la raíz del repo
    os.makedirs(f"{ruta}/data", exist_ok=True)
    return f"{ruta}/data"

def guardar_gold(df, nombre):
    df.write.mode("overwrite").option("overwriteSchema", True).saveAsTable(f"workspace.pobreza.{nombre}_gold")
    df.toPandas().to_json(f"{carpeta_data()}/{nombre}.json", orient="records", force_ascii=False, indent=1)
    print("OK:", nombre)

In [0]:
gold = spark.table("workspace.pobreza.banco_mundial_silver").orderBy("indicador", "iso3", "anio")
guardar_gold(gold, "comparacion_paises")

# Último dato de cada país, para revisar rápido
from pyspark.sql.window import Window
w = Window.partitionBy("indicador", "iso3").orderBy(F.desc("anio"))
display(gold.withColumn("r", F.row_number().over(w)).filter("r = 1").drop("r").orderBy("indicador", "valor"))

## Hallazgos (último dato disponible)

- **Desempleo juvenil:** Uruguay tiene **25,2%** (2025), el más alto del grupo. El promedio de América Latina es 12,2% y el de la OCDE 11,6%.
- **Educación:** solo el **35%** de los uruguayos de 25 años o más terminó la secundaria (2024), el valor más bajo de los 12 países. En Chile es 70% y en Brasil 62%.
- **Homicidios:** 11,3 cada 100.000 (2023). Es menos que Brasil o México, pero más del doble que Argentina (4,5) y bastante más que Chile (6,4) y Paraguay (6,8).
- **PIB per cápita:** es el más alto del grupo (US$ 32.742 PPA), pero es el 60% del de la OCDE.
- **Lo bueno:** la desigualdad más baja del grupo (Gini 40) y una pobreza (línea US$ 8,30) de 5,9%. Chile tiene una pobreza más baja (4,5%).